# Mixture of Experts (MoE)
## Sparse Gating, Load Balancing, and Scaling Laws

**MLNN Tutorial · University of Hertfordshire · 2025**  
GitHub: https://github.com/revtheundefined/moe-tutorial

### Resources used
- Jacobs, R.A. et al. (1991) 'Adaptive mixtures of local experts', *Neural Computation*, 3(1). https://doi.org/10.1162/neco.1991.3.1.79
- Shazeer, N. et al. (2017) 'Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer'. https://arxiv.org/abs/1701.06538
- Fedus, W., Zoph, B. and Shazeer, N. (2022) 'Switch Transformers: Scaling to Trillion Parameter Models'. https://arxiv.org/abs/2101.03961
- Du, N. et al. (2022) 'GLaM: Efficient Scaling of Language Models with Mixture-of-Experts'. https://arxiv.org/abs/2112.06905
- Jiang, A.Q. et al. (2024) 'Mixtral of Experts'. https://arxiv.org/abs/2401.04088
- Zoph, B. et al. (2022) 'ST-MoE: Designing Stable and Transferable Sparse Expert Models'. https://arxiv.org/abs/2202.08906

### Accessibility
All figures use colourblind-safe teal/amber palettes. Heatmaps use magma colourmap. Alt-text in every Markdown cell.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'torch', 'matplotlib', 'numpy', 'scipy', '--quiet'])

In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.optim as optim
from scipy.ndimage import uniform_filter1d
import warnings; warnings.filterwarnings('ignore')

BG='#080F0E'; CARD='#0D1A18'; GRID='#162B28'
TEAL='#2DD4BF'; AMBR='#FBBF24'; SAGE='#86EFAC'
SKY='#7DD3FC'; ROSE='#FB7185'; LAVD='#C084FC'
TEXT='#F0FDFA'; MUTE='#5EEAD4'

plt.rcParams.update({'figure.facecolor':BG,'axes.facecolor':CARD,
                     'font.family':'monospace','font.size':10})
def sa(ax):
    ax.set_facecolor(CARD); ax.tick_params(colors=MUTE,labelsize=9)
    ax.xaxis.label.set_color(TEXT); ax.yaxis.label.set_color(TEXT); ax.title.set_color(TEXT)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID)
    ax.grid(True,color=GRID,linewidth=0.7,alpha=0.8)

torch.manual_seed(42); np.random.seed(42)
print('Setup complete.')

---
## Section 1: MoE Architecture — Sparse Top-K Gating

A Mixture of Experts layer replaces a single FFN with N expert FFNs and a **gating network** that routes each token to the top-K experts. Only K/N of the experts compute for each token, giving a massive parameter count with controlled compute.

Output = Σ_{i ∈ TopK} G_i(x) · E_i(x)

**Alt-text:** Full MoE layer implementation with Top-K sparse gating and load balancing auxiliary loss.

In [ ]:
class Expert(nn.Module):
    """Single expert: a standard FFN (feed-forward network)."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )
    def forward(self, x): return self.net(x)


class MoELayer(nn.Module):
    """
    Sparse Mixture of Experts Layer (Shazeer et al., 2017).
    
    Architecture:
        - N expert FFNs, each identical in architecture
        - 1 gating network: linear + softmax
        - Top-K routing: only K experts compute per token
        - Load balancing loss prevents expert collapse
    
    Training objective:
        L_total = L_task + alpha * L_balance
    where:
        L_balance = N * Σ_i f_i * p_i
        f_i = fraction of tokens routed to expert i (hard)
        p_i = mean gating probability for expert i (soft, differentiable)
    """
    def __init__(self, d_model, d_ff, n_experts=8, top_k=2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        
        # Gating network: maps input to expert logits
        self.gate = nn.Linear(d_model, n_experts, bias=False)
        
        # Expert pool
        self.experts = nn.ModuleList([
            Expert(d_model, d_ff) for _ in range(n_experts)
        ])
    
    def forward(self, x):
        """
        x: (batch_size, d_model)
        returns: (output, load_balance_loss)
        """
        B, d = x.shape
        
        # Step 1: Compute gating logits and select top-K
        logits = self.gate(x)                                      # (B, E)
        topk_logits, topk_idx = torch.topk(logits, self.top_k, dim=-1)  # (B, K)
        gate_weights = torch.softmax(topk_logits, dim=-1)          # (B, K) — renormalise
        
        # Step 2: Compute expert outputs for selected experts only
        d_out = self.experts[0].net[-1].out_features
        output = torch.zeros(B, d_out, device=x.device)
        
        for k in range(self.top_k):
            for e in range(self.n_experts):
                mask = (topk_idx[:, k] == e)           # which tokens go to expert e at slot k
                if mask.any():
                    expert_out = self.experts[e](x[mask])          # only compute for routed tokens
                    output[mask] += gate_weights[mask, k:k+1] * expert_out
        
        # Step 3: Load balancing loss (Fedus et al., 2022 / ST-MoE)
        # Differentiable via soft gates; scale by N so loss is O(1)
        soft_gates = torch.softmax(logits, dim=-1)                 # (B, E) — soft version
        tokens_per_expert = soft_gates.mean(dim=0)                 # (E,) — mean routing prob
        # L_balance = N * Σ_i p_i² (encourages uniform distribution)
        load_balance_loss = self.n_experts * (tokens_per_expert ** 2).sum()
        
        return output, load_balance_loss


# Sanity check
moe = MoELayer(d_model=64, d_ff=128, n_experts=8, top_k=2)
x_test = torch.randn(32, 64)
out, lb = moe(x_test)
print(f'Input:  {x_test.shape}')
print(f'Output: {out.shape}')
print(f'Load balance loss: {lb.item():.4f}')
print(f'Ideal load balance loss: {1.0:.4f} (all experts equally used → p_i=1/N → N*Σ(1/N²)=1)')
total_params = sum(p.numel() for p in moe.parameters())
active_params = sum(p.numel() for p in moe.gate.parameters()) + \
                2 * sum(p.numel() for p in moe.experts[0].parameters())
print(f'Total params: {total_params:,} | Active per token: {active_params:,} ({active_params/total_params*100:.1f}%)')

---
## Section 2: Load Balancing — Preventing Expert Collapse

Without load balancing, the gating network collapses: a few experts get all tokens, others are never trained. The **auxiliary load balancing loss** (Fedus et al., 2022) prevents this:

L_balance = N · Σᵢ fᵢ · pᵢ

where fᵢ = fraction of tokens routed to expert i (hard), pᵢ = mean soft gating probability for expert i.

**Alt-text:** Demonstration of expert collapse without load balancing vs. balanced routing with auxiliary loss.

In [ ]:
def train_moe(use_load_balancing=True, n_steps=300, alpha=0.01):
    """
    Train a MoE on structured regression task.
    Returns: (losses, expert_utilisation_history)
    """
    torch.manual_seed(42)
    D_in, D_out, N = 16, 4, 800
    
    # Data: 4 regions, each requires different linear map
    X = torch.randn(N, D_in)
    y = torch.zeros(N, D_out)
    for i in range(N):
        region = torch.argmax(X[i, :4]).item()
        y[i] = X[i, 4:8] * [1, -1, 0.5, -0.5][region]
    
    model = MoELayer(d_model=D_in, d_ff=32, n_experts=8, top_k=2)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    
    losses, utilisation = [], []
    
    for step in range(n_steps):
        idx = torch.randint(0, N, (128,))
        xb, yb = X[idx], y[idx]
        
        out, lb_loss = model(xb)
        task_loss = ((out - yb) ** 2).mean()
        
        if use_load_balancing:
            total_loss = task_loss + alpha * lb_loss
        else:
            total_loss = task_loss
        
        opt.zero_grad(); total_loss.backward(); opt.step()
        losses.append(task_loss.item())
        
        # Track expert utilisation
        if step % 20 == 0:
            with torch.no_grad():
                gate_logits = model.gate(X)
                top_idx = torch.topk(gate_logits, 2, dim=-1).indices
                util = [(top_idx == e).float().sum().item() / (N * 2)
                        for e in range(8)]
                utilisation.append(util)
    
    return losses, utilisation

print('Training without load balancing...')
losses_no_lb, util_no_lb = train_moe(use_load_balancing=False)
print('Training with load balancing...')
losses_lb, util_lb = train_moe(use_load_balancing=True)

print(f'\nFinal task loss without LB: {losses_no_lb[-1]:.4f}')
print(f'Final task loss with LB:    {losses_lb[-1]:.4f}')
print()
print('Expert utilisation (last checkpoint):')
print('Without LB:', [f'{v:.2f}' for v in util_no_lb[-1]])
print('With LB:   ', [f'{v:.2f}' for v in util_lb[-1]])
ideal = 1.0 / 8 * 2  # 2 experts per token, 8 experts total = 0.25 each
print(f'Ideal (balanced): [{ideal:.2f}] × 8')

---
## Summary Table

| Model | Top-K | Routing | Load Balancing | Used in |
|-------|-------|---------|----------------|----------|
| Jacobs 1991 | Soft | Learnable | None | Original MoE |
| Shazeer 2017 | Top-2 | Noisy | Aux loss | Sparse MoE |
| Switch Transformer (2021) | Top-1 | Learned | Aux loss | T5-MoE |
| GLaM (2022) | Top-2 | Learned | Aux loss | Google GLaM |
| Mixtral (2024) | Top-2 | Learned | Aux loss | Open source |

## References

1. Jacobs, R.A. et al. (1991) 'Adaptive mixtures of local experts', *Neural Computation*, 3(1). https://doi.org/10.1162/neco.1991.3.1.79
2. Shazeer, N. et al. (2017) 'Outrageously Large Neural Networks: The Sparsely-Gated MoE Layer'. https://arxiv.org/abs/1701.06538
3. Fedus, W. et al. (2022) 'Switch Transformers: Scaling to Trillion Parameter Models'. https://arxiv.org/abs/2101.03961
4. Du, N. et al. (2022) 'GLaM: Efficient Scaling of Language Models with MoE'. https://arxiv.org/abs/2112.06905
5. Jiang, A.Q. et al. (2024) 'Mixtral of Experts'. https://arxiv.org/abs/2401.04088
6. Zoph, B. et al. (2022) 'ST-MoE: Designing Stable and Transferable Sparse Expert Models'. https://arxiv.org/abs/2202.08906